In [1]:
import os
import pandas as pd
import numpy as np
import json
from huggingface_hub import login

# Set environment variable for TabPFN downloader
os.environ["HF_TOKEN"] = "REDACTED_HF_TOKEN"
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6474821bd815855e4ef484df', 'name': 'faisalomari', 'fullname': 'Faisal Omari', 'isPro': False, 'avatarUrl': '/avatars/0e758a783d5540170b3899e9d3631026.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'arabic_majaz', 'role': 'fineGrained', 'createdAt': '2026-01-21T19:34:04.770Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '6474821bd815855e4ef484df', 'type': 'user', 'name': 'faisalomari'}, 'permissions': ['repo.content.read', 'repo.write', 'user.webhooks.read', 'user.webhooks.write', 'discussion.write']}]}}}}


In [3]:
DATA_PATH = r"embeddings_out/dataset_with_embeddings.parquet"
df = pd.read_parquet(DATA_PATH)

if df["embedding"].dtype == object and isinstance(df["embedding"].iloc[0], str):
    df["embedding"] = df["embedding"].apply(json.loads)

X = np.vstack(df["embedding"].values).astype(np.float32)
y = df["label_majaz"].astype(int).values

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 1. Scale the data
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 2. Apply PCA to reduce dimensions from 3072 to 512
print("Running PCA dimensionality reduction...")
pca = PCA(n_components=512, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)

print(f"Original shape: {X_train_s.shape}")
print(f"PCA shape:      {X_train_pca.shape}")

Running PCA dimensionality reduction...
Original shape: (779, 3072)
PCA shape:      (779, 512)


In [4]:
from tabpfn import TabPFNClassifier

# We no longer need ignore_pretraining_limits=True because 512 < 2000
model = TabPFNClassifier(device='cpu')

# TabPFN works best with a manageable training set size. 
# If it's still slow, use X_train_pca[:2000]
model.fit(X_train_pca, y_train)

print("Model trained successfully with PCA features!")

/home/faisal/Desktop/poems/.my-env/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 1060 6GB which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/faisal/Desktop/poems/.my-env/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/faisal/Desktop/poems/.my-env/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
NVIDIA GeForce GTX 1060 6GB with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce GTX 1060 6GB GPU with PyTorch, please check the instructions at https

Model trained successfully with PCA features!


In [5]:
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support, roc_auc_score

batch_size = 32 
n_samples = X_test_pca.shape[0]
all_preds = []
all_probs = []

print(f"Predicting {n_samples} samples in batches...")
for i in tqdm(range(0, n_samples, batch_size)):
    X_batch = X_test_pca[i : i + batch_size]
    
    batch_pred = model.predict(X_batch)
    batch_prob = model.predict_proba(X_batch)
    
    all_preds.append(batch_pred)
    all_probs.append(batch_prob)

y_pred = np.concatenate(all_preds)
y_probs = np.concatenate(all_probs)

# Metrics
acc = accuracy_score(y_test, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")
auc = roc_auc_score(y_test, y_probs[:, 1])

print(f"\nAccuracy: {acc:.4f} | F1: {f1:.4f} | ROC AUC: {auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

Predicting 335 samples in batches...


  0%|          | 0/11 [00:00<?, ?it/s]


Accuracy: 0.6746 | F1: 0.6727 | ROC AUC: 0.7189

Classification Report:
               precision    recall  f1-score   support

           0     0.6746    0.6786    0.6766       168
           1     0.6747    0.6707    0.6727       167

    accuracy                         0.6746       335
   macro avg     0.6746    0.6746    0.6746       335
weighted avg     0.6746    0.6746    0.6746       335



In [6]:
import joblib
from pathlib import Path

out_dir = Path("majaz_embedding_classifier_tabpfn")
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, out_dir / "tabpfn_model.joblib")
joblib.dump(scaler, out_dir / "scaler.joblib")
joblib.dump(pca, out_dir / "pca.joblib") # <--- Save PCA here

print(f"All components saved to: {out_dir.resolve()}")

All components saved to: /home/faisal/Desktop/poems/Majaz/majaz_embedding_classifier_tabpfn
